# Projekt C — obsługa hotelu i pakietów wypoczynkowych
## Specyfikacja i plan realizacji do wspólnej rewizji

**Wersja:** 0.2 • **Data:** 11.10.2026 • **Status:** do przeglądu zespołu  
**Koordynator:** Adrian Zieliński • **Zespół:** Projekt C

Tworzymy system dla jednego hotelu. Klient zakłada konto, sprawdza dostępność i rezerwuje pokój z opcjonalnym pakietem. Personel obsługuje pobyt, a administrator zarządza ofertą, kontami oraz integracjami. Inne projekty korzystają z konfigurowalnego API, które pozwala także tworzyć, zmieniać i anulować rezerwacje.

Ten notebook służy do uzgodnienia zakresu i zasad działania przed implementacją. Nie zawiera działającej aplikacji. Ustalenia techniczne są propozycją do zatwierdzenia, a nie potwierdzeniem wykonanych funkcji. Rzeczywiste kontrakty A/B/D wymagają uzgodnienia z partnerami.

**Podstawa planu:** prezentacja 1, szczególnie slajdy 4, 6–7, 11, 13–19; prezentacja 2, szczególnie slajdy 4 i 8–13; wspólny scenariusz dostarczony na zdjęciu; skład zespołu dostarczony przez użytkownika. Szczegółowe reguły poniżej, których nie określają slajdy, są propozycjami implementacyjnymi do akceptacji.

Prezentacja 1 miejscami używa określenia „trzy aplikacje”, ale wymienia cztery projekty i opisuje scenariusz A/B/C/D. Plan przyjmuje cztery niezależne aplikacje; implementujemy wyłącznie C i jego klientów API. Scenariusz wspólny ma pierwszeństwo przed wcześniejszą propozycją ograniczenia integracji do katalogów ofert i rabatu kwotowego.


## 1 Jak pracować nad dokumentem w Google Colab

1. Zaimportuj plik `.ipynb` do Colab przez **Plik → Prześlij notatnik** i zapisz kopię na Dysku Google.
2. Udostępnij zapisany notebook zespołowi. Przyznaj możliwość edycji osobom, które mają aktualizować treść i tabele; osoby opiniujące mogą korzystać z komentarzy.
3. Sekcje są komórkami tekstowymi. Aby poprawić treść, otwórz komórkę do edycji i zmień Markdown. Nie trzeba uruchamiać środowiska Python.
4. Każda uwaga powinna odwoływać się do numeru sekcji lub identyfikatora wymagania, np. `F-04`, i zawierać konkretną propozycję zmiany.
5. Zgłoszenia wpisujemy do rejestru uwag w sekcji 16. Koordynator zapisuje uzgodnione decyzje i numer nowej wersji w sekcji 17.
6. Każdy członek zespołu osobiście uzupełnia tabelę akceptacji w sekcji 18. Nie zatwierdzamy za inne osoby.

**Zasada wspólnej edycji:** unikamy równoczesnego zmieniania tej samej komórki. Najpierw komentarz lub wpis w rejestrze, potem uzgodniona poprawka treści. Korzystamy z historii zmian notebooka.

**Akceptacja wersji:** wszyscy mają wpisane „akceptuję” dla tej samej wersji i wszystkie uwagi oznaczone jako blokujące są rozwiązane. Po istotnej zmianie zakresu, API, bezpieczeństwa lub reguł biznesowych zainteresowane osoby ponawiają akceptację nowej wersji; finalny zakres potwierdza ponownie cały zespół.

Do dokumentu wpisujemy wyłącznie dane demonstracyjne. Klucze API, hasła i rzeczywiste dane gości przechowujemy poza notebookiem.


## 2 Zakres i priorytety

**Cel odbioru:** samodzielna obsługa klientów hotelu oraz udział C w pełnym scenariuszu voucher → pobyt → masaż i trening → wykonanie usług → częściowe wykorzystanie vouchera.

| ID | Funkcja | Warunek ukończenia |
|---|---|---|
| F-01 | Konta i role | Rejestracja, logowanie, profil, hasła, blokady; brak samodzielnego nadania roli personelu |
| F-02 | Oferta pokoi | Numer, typ, pojemność, wyposażenie, cena za noc i aktywność |
| F-03 | Dostępność i wycena | Wolne pokoje w okresie, poprawna liczba nocy i cena |
| F-04 | Rezerwacja pobytu | Własny klient, recepcja lub D; bez podwójnej rezerwacji |
| F-05 | Zmiana/anulowanie | Dostęp zależny od właściciela, źródła, stanu i powiązanych usług |
| F-06 | Obsługa pobytu | Goście, statusy, zameldowanie i wymeldowanie |
| F-07 | Pakiety | Nocleg z dodatkami A/B; nazwa, cena, oferta partnera i liczba przysługujących usług |
| F-08 | Własne API | Oferta, dostępność, utworzenie, odczyt, zmiana i anulowanie pobytu; odczyt planu |
| F-09 | Konfiguracja | Panel adresów, trybów, danych dostępowych i uprawnień partnerów |
| F-10 | Historia | Autor, czas, obiekt i wynik istotnej operacji |
| F-11 | Rezerwacje A/B | Odczyt ofert/terminów, zamówienie, potwierdzenie, odczyt stanu i anulowanie |
| F-12 | Voucher D | Sprawdzenie hotelowej części i potwierdzenie jej wykonania bez konsumowania części A/B |
| F-13 | Pulpit i plan pobytu | Przyjazdy/wyjazdy oraz harmonogram usług zewnętrznych |

F-01–F-13 są zakresem docelowym. Kolejność implementacji: rdzeń hotelowy, własne API, rezerwacje A/B, voucher D, wspólny odbiór. Symulatory umożliwiają pracę przed udostępnieniem API, ale końcowa demonstracja powinna używać rzeczywistych systemów partnerów. Minimum „jeden klient API” ze slajdów nie zastępuje pełnego scenariusza A/B/D wymaganego dla C.

**Poza wersją 1:** rzeczywiste płatności, faktury i rozliczanie partnerów w C, wiele hoteli, aplikacja mobilna, wspólne konta/SSO, automatyczny optymalizator terminów, samodzielne odzyskiwanie hasła przez e-mail. Sprzedaż voucherów i zestawienia rozliczeń należą do D; grafiki pracowników/sal i wykonanie masażu/treningu należą do A/B.


## 3 Użytkownicy i uprawnienia

| Operacja | Klient | Recepcjonista | Administrator | Partner API |
|---|---|---|---|---|
| Odczyt aktywnej oferty i dostępności | Tak | Tak | Tak | Z uprawnieniem |
| Utworzenie rezerwacji | Dla siebie | Dla klienta/gościa | Dla klienta/gościa | Z uprawnieniem |
| Odczyt rezerwacji | Własne | Wszystkie | Wszystkie | Tylko utworzone przez siebie |
| Zmiana/anulowanie | Własne, przed przyjazdem | Wszystkie potwierdzone | Wszystkie potwierdzone | Własne, z uprawnieniem, przed przyjazdem |
| Zameldowanie/wymeldowanie | Nie | Tak | Tak | Nie |
| Zarządzanie pokojami/pakietami | Nie | Odczyt | Tak | Nie |
| Zmiana ról, blokada kont | Nie | Nie | Tak | Nie |
| Konfiguracja integracji i kluczy | Nie | Nie | Tak | Nie |

Gość bez konta może przeglądać ofertę. Rezerwacja przez stronę wymaga konta. Rezerwacja recepcji lub partnera może zawierać dane gościa bez zakładania mu konta. Partner nie otrzymuje dostępu do kont klientów ani globalnej listy rezerwacji.

Pierwszy administrator powstaje przez skrypt inicjalizacji z hasłem przekazanym konfiguracją środowiska. Administrator tworzy konta personelu. Nie można zablokować lub zdegradować ostatniego aktywnego administratora.

Blokada konta kończy jego sesje i uniemożliwia logowanie; istniejące rezerwacje pozostają w systemie. Zmiana hasła unieważnia pozostałe sesje. Administrator ustawia hasło tymczasowe przy resecie, a użytkownik musi je zmienić przy następnym logowaniu.


## 4 Scenariusze biznesowe i wspólna realizacja

### S-01 Własny klient hotelu

Klient rejestruje konto, wybiera daty, pokój i ewentualnie pakiet. Otrzymuje podsumowanie, potwierdza cenę, a C ponownie sprawdza dostępność w transakcji. Powstaje potwierdzony pobyt i kopia jego dodatków. Rezerwacja hotelowa nie oznacza jeszcze potwierdzenia usług A/B. Recepcja uzgadnia z klientem plan, a następnie rezerwuje konkretne terminy przez API.

### S-02 Zmiana i anulowanie

Klient lub partner źródłowy może przed przyjazdem zmienić daty/liczbę gości własnego potwierdzonego pobytu. Nowa cena wymaga akceptacji. Jeśli pobyt ma potwierdzone, oczekujące lub niepewne usługi A/B, zmiana terminów jest zablokowana do ich wyjaśnienia/anulowania przez recepcję. Nie wykonujemy ukrytej automatycznej zmiany wszystkich usług.

Anulowanie pobytu bez usług zewnętrznych zwalnia pokój od razu. Przy usługach A/B C najpierw anuluje je u partnerów. Dopiero po jednoznacznym potwierdzeniu wszystkich anulowań anuluje hotel. Częściowe niepowodzenie pozostawia pobyt hotelowy aktywny i pokazuje, które usługi już anulowano, a które wymagają działania. Wykorzystana hotelowa część vouchera blokuje zwykłe anulowanie; zwroty są poza wersją 1.

### S-03 Wspólny scenariusz ze slajdu 13

| Krok | System i działanie | Odpowiedzialność C |
|---|---|---|
| 1 | Klient kupuje w D voucher: nocleg, masaż i trening | C udostępnia D swoją ofertę; sprzedaż i płatność są w D |
| 2 | D sprawdza dostępność i rezerwuje pobyt w C | C sprawdza pokój, zapisuje kontakt, pakiet, źródło D i referencję vouchera; zwraca ID |
| 3 | C uzgadnia z klientem plan | Recepcja wybiera terminy usług podczas pobytu z dostępności A/B |
| 4 | C rezerwuje masaż w A i trening w B | Wywołuje oba API, zapisuje ID operacji i potwierdzenia rezerwacji |
| 5 | A/B potwierdzają i blokują terminy | C pokazuje potwierdzony plan lub wskazuje niepotwierdzone pozycje |
| 6 | Każdy usługodawca sprawdza voucher w D przy realizacji | C sprawdza uprawnienie do noclegu przed zameldowaniem; A/B sprawdzają swoje części samodzielnie |
| 7 | Usługa zostaje wykonana i lokalnie zakończona | C wymeldowuje gościa i potwierdza w D wykonanie hotelowej części; A/B potwierdzają swoje |
| 8 | D pokazuje pozostałe części i zestawienie rozliczeń | C udostępnia stan własnej rezerwacji; nie tworzy własnego systemu rozliczeń D |

### S-04 Koordynacja usług

Recepcja otwiera plan, wybiera dodatek, pobiera wolne terminy, uzgadnia jeden z klientem i wysyła rezerwację. Cały czas trwania usługi musi mieścić się w pobycie. Potwierdzenie partnera zapisuje zewnętrzne ID oraz datę/godzinę. Odmowa daje wybór innego terminu; timeout po wysłaniu daje status wymagający sprawdzenia wyniku, a nie automatyczny sukces.

### S-05 Obsługa hotelowej części vouchera

Przy pobycie z voucherem recepcja przed zameldowaniem sprawdza w D kod, ważność, część noclegową, uprawnionego dostawcę C i powiązanie z rezerwacją. Brak możliwości potwierdzenia nie oznacza ważnego vouchera — pobyt zostaje potwierdzony w hotelu, ale zameldowanie wymaga wyjaśnienia. Po wymeldowaniu C zapisuje lokalne wykonanie i zgłasza do D wyłącznie nocleg. Nie zużywa masażu ani treningu.

### S-06 Własny klient bez vouchera

Hotel działa samodzielnie: rezerwacja i obsługa zwykłego pobytu nie wymagają D. Pakiet zakupiony bez vouchera może korzystać z rezerwacji A/B, ale nie uruchamia potwierdzania wykorzystania w D. Płatności i rozrachunki tych usług nie są automatyzowane w C.


## 5 Reguły rezerwacji i wyceny

| Reguła | Ustalenie dla wersji 1 |
|---|---|
| Hotel i waluta | Jeden hotel, PLN; daty biznesowe według Europe/Warsaw |
| Daty | Format `YYYY-MM-DD`; przyjazd wcześniejszy niż wyjazd; nowa rezerwacja nie może zaczynać się w przeszłości |
| Zajętość | Przedział `[przyjazd, wyjazd)`; wyjazd i kolejny przyjazd tego samego dnia są dozwolone |
| Konflikt | Istniejąca rezerwacja koliduje, jeśli jej przyjazd < nowy wyjazd i jej wyjazd > nowy przyjazd |
| Statusy zajmujące termin | `confirmed`, `checked_in`, `completed`; `cancelled` nie zajmuje terminu |
| Goście | Liczba całkowita co najmniej 1 i nie większa niż pojemność pokoju |
| Pakiet | Co najwyżej jeden pakiet na rezerwację; cena za cały pobyt, dodatki według opisu pakietu |
| Cena | Liczba nocy × stawka pokoju + cena pakietu; kwoty jako całkowite grosze |
| Historia ceny | Rezerwacja przechowuje stawkę, cenę pakietu i sumę; późniejsza edycja oferty nie zmienia starych rezerwacji |
| Zmiana terminu | Ponowna wycena według aktywnej oferty; zmiana wymaga potwierdzenia nowej ceny |
| Wyłączenie oferty | Blokuje nowe rezerwacje; istniejące pozostają ważne i wymagają obsługi personelu |

Dozwolone przejścia statusów: `confirmed → cancelled`, `confirmed → checked_in`, `checked_in → completed`. Status nie jest dowolnym polem formularza lub `PATCH` API.

Sprawdzenie dostępności i zapis muszą być atomowe. Przy SQLite używamy transakcji zapisu z uzyskaniem blokady przed odczytem zajętości. Sam wcześniejszy odczyt dostępności nie wystarcza. Ograniczone ponowienie transakcji obsługuje chwilową blokadę bazy; utrzymująca się blokada zwraca błąd tymczasowy bez częściowego zapisu. Test równoległy musi potwierdzić brak podwójnej rezerwacji.

Historia ma charakter operacyjny: zapisujemy autora, czas UTC, typ operacji, identyfikator obiektu oraz bezpieczny opis zmiany. Nie zapisujemy haseł, kluczy ani całych poufnych żądań.


**Godziny i plan pobytu:** daty pokoi pozostają przedziałami dni. Przyjmujemy zameldowanie od 14:00 i wyjazd do 11:00, z możliwością ustawienia tych godzin w konfiguracji hotelu. Termin usługi ma timestamp ISO 8601 ze strefą/offsetem i koniec; cała usługa mieści się między tymi godzinami pobytu. C pilnuje również, aby masaż i trening jednego gościa nie nakładały się. Ostateczną dostępność zasobów A/B sprawdza partner.

**Goście i jednostki usług:** osoba rezerwująca przekazuje imię i nazwisko każdego gościa; dane kontaktowe należą do osoby kontaktowej. Dodatek ma liczbę jednostek i przypisanie do gościa. Każda jednostka masażu/treningu ma osobną pozycję planu. W pierwszej demonstracji używamy prostego pakietu: jeden pobyt, jeden masaż, jeden trening.

**Pakiet a voucher:** pakiet własny jest ofertą cenową C. Voucher z D potwierdza prawo do określonych usług, a nie automatyczny rabat kwotowy od całego pobytu. Rezerwacja przechowuje cenę referencyjną hotelu i źródło finansowania `direct`/`voucher`; C nie pobiera ponownie zapłaty od posiadacza vouchera ani nie implementuje rozliczenia partnerów.

Zmiana ceny/terminu przy voucherze wymaga zgodności zakresu i ważności potwierdzonej przez D. Jeśli wymaga nowej wyceny po stronie D, zwykły PATCH zwraca konflikt wymagający uzgodnienia. Wycofanie/usunięcie vouchera obsługuje D, nie dowolna edycja pola w C.


## 6 Ekrany aplikacji

| Ekran | Co użytkownik robi | Ważne zachowanie |
|---|---|---|
| Oferta i wyszukiwarka | Wpisuje daty/osoby, wybiera pokój i pakiet | Pusty wynik jest czytelnym stanem, nie błędem |
| Rejestracja i logowanie | Tworzy konto lub rozpoczyna sesję | Walidacja i komunikat błędu; brak wyboru roli |
| Podsumowanie rezerwacji | Sprawdza cenę i potwierdza zapis | Blokada wielokrotnego kliknięcia; obsługa konfliktu |
| Moje rezerwacje | Przegląda szczegóły, zmienia termin, anuluje | Przyciski zależą od statusu i daty |
| Profil | Aktualizuje dane i hasło | Sprawdzenie aktualnego hasła przy jego zmianie |
| Pulpit recepcji | Widzi przyjazdy/wyjazdy i wyszukuje pobyt | Filtrowanie po dacie, statusie i numerze |
| Szczegóły pobytu | Zarządza gośćmi, melduje i wymeldowuje | Weryfikacja części hotelowej vouchera |
| Plan pobytu | Wybiera terminy A/B, rezerwuje, odświeża i anuluje usługi | ID partnera, stan potwierdzenia, częściowe błędy |
| Oferta administratora | Edytuje pokoje i pakiety | Dezaktywacja zamiast usuwania historii |
| Konta administratora | Tworzy personel, zmienia role, blokuje/resetuje | Ochrona ostatniego administratora |
| Integracje i API | Konfiguruje A/B/D, klucze i uprawnienia | Sekrety zamaskowane; klucz widoczny raz po utworzeniu |
| Historia | Przegląda istotne operacje | Filtry po obiekcie, autorze i czasie |

Każdy formularz ma stan ładowania, walidację i komunikat wyniku. Błędy przy polach nie kasują wprowadzonych danych. Widoki mają działać na laptopie i telefonie, mieć etykiety pól oraz obsługę klawiatury. Backend weryfikuje wszystkie dane niezależnie od formularza.

Klient i D widzą plan pobytu w zakresie własnej rezerwacji; operacje rezerwowania/anulowania A/B wykonuje personel C. Przycisk odświeżenia pobiera aktualny stan rezerwacji od partnera. Nie polegamy wyłącznie na ręcznie wpisanym „wykonano” dla usług A/B.


## 7 Architektura i organizacja kodu

**Frontend:** React, Vite, TypeScript, React Router. **Backend:** Express, TypeScript, Prisma i SQLite. **Testy:** Vitest, Supertest oraz Playwright dla kilku kluczowych scenariuszy. **Dokumentacja API:** OpenAPI i Swagger UI.

```text
Przeglądarka → React → /api/v1 → Express → logika biznesowa → SQLite
Partner A/B/D → klucz API → /api/v1 → ta sama logika rezerwacji
Express → adapter A/B/D → rzeczywiste API lub symulator
```

Jedno repozytorium: `frontend/`, `backend/`, `docs/`. Backend oddziela kontrolery HTTP, logikę biznesową, dostęp do danych i adaptery partnerów. Nie budujemy mikroserwisów ani kolejki zadań.

W trybie developerskim Vite przekazuje `/api` do backendu. W wersji demonstracyjnej backend udostępnia również zbudowany frontend pod jednym adresem. Środowisko zawiera `DATABASE_URL`, sekret sesji, klucz szyfrowania konfiguracji i dane początkowego administratora. Repo zawiera `.env.example` bez sekretów.

Skrypty docelowe: `npm run setup` (zależności, migracje, dane demo), `npm run dev`, `npm run build`, `npm start`, `npm test`, `npm run test:e2e`. Skrypt zasilający bazę danymi demonstracyjnymi jest idempotentny; nie usuwa istniejących rezerwacji.

Sesje przechowujemy po stronie serwera. Ciasteczko jest HttpOnly i SameSite, a w środowisku HTTPS także Secure. Zmieniające dane operacje sesyjne wymagają tokena CSRF. Klucze partnerów zastępują sesję tylko na dopuszczonych endpointach partnerskich. Logowanie i publiczne endpointy mają ograniczenie częstotliwości żądań.

Pierwsza wersja pracuje na pojedynczej instancji backendu. Przy wdrożeniu publicznym używamy HTTPS i trwałego dysku dla bazy. Kopia bazy jest tworzona mechanizmem bezpiecznej kopii SQLite, a nie przez kopiowanie aktywnego pliku bez uwzględnienia dziennika.


## 8 Model danych

| Obiekt | Najważniejsze informacje | Relacje i ograniczenia |
|---|---|---|
| User | email, hash hasła, imię/nazwisko, rola, aktywność, wymuszona zmiana hasła | Unikalny znormalizowany email; wiele rezerwacji |
| Room | numer, typ, wyposażenie, pojemność, cena za noc w groszach, aktywność | Unikalny numer; wiele rezerwacji |
| Package | nazwa, opis, cena w groszach, aktywność | Lista dodatków; wiele rezerwacji |
| PackageItem | nazwa, rodzaj HOTEL/A/B, referencja oferty partnera, liczba jednostek | Należy do pakietu; brak automatycznego terminu |
| Reservation | numer, daty, goście, status, kontakt gościa, ceny, wersja | Pokój; opcjonalnie konto, pakiet i partner źródłowy |
| ReservationItem | kopia dodatku, status realizacji, notatka | Zachowuje treść pakietu z dnia rezerwacji |
| Integration | partner A/B/D, adres, tryb, aktywność, timeout, zaszyfrowane dane dostępowe | Konfiguracja adaptera; nie klucz wejściowy C |
| PartnerApiKey | nazwa partnera, hash klucza, uprawnienia, aktywność, data użycia | Klucz identyfikuje partnera i jego operacje |
| IdempotencyRecord | partner, klucz żądania, skrót treści, wynik | Chroni tworzenie przed duplikacją |
| VoucherOperation | rezerwacja, ID operacji, chroniony kod, ID części hotelowej, status | Brak dublowania wykorzystania; wynik niepewny zachowany |
| AuditEvent | autor użytkownik/partner, czas, operacja, obiekt, opis | Rejestr zmian bez sekretów |
| Session | identyfikator sesji, użytkownik, termin ważności | Możliwość unieważnienia sesji |

Powiązanie partnera z rezerwacją wyznacza serwer. Nie przyjmujemy `partnerId`, roli ani autora operacji z danych klienta. Usunięcie konta i pełna polityka retencji są poza wersją 1; do demonstracji używamy danych fikcyjnych.


### Uzupełnienie dla wspólnego scenariusza

| Obiekt | Dane potrzebne do integracji |
|---|---|
| ReservationGuest | Imię, nazwisko i przypisanie do rezerwacji; pozycje usług przypisane do gościa |
| PartnerServiceBooking | Pozycja planu, A/B, ID oferty, gość, początek/koniec, ID zewnętrznej rezerwacji, status, ID żądania |
| VoucherBinding | Referencja vouchera D, część hotelowa, źródło D, stan ostatniej weryfikacji; chroniony kod jeśli wymagany |
| PartnerOperation | Utworzenie/anulowanie/potwierdzenie, ID operacji, stan pending/succeeded/rejected/unknown, ostatni wynik |
| HotelSettings | Godzina przyjazdu/wyjazdu i strefa Europe/Warsaw |

VoucherOperation oznacza teraz weryfikację/potwierdzenie **hotelowej części usługi**, nie rabat kwotowy. ReservationItem zachowuje kopię liczby i rodzaju usług pakietu oraz referencję pozycji vouchera. Całe dane sprzedaży i stan wszystkich części pozostają w D. PartnerServiceBooking przechowuje referencje, nie kopię grafików A/B. C nigdy nie łączy się bezpośrednio z bazami partnerów.


## 9 Kontrakt API systemu C

Wspólny prefiks: `/api/v1`. Żądania i odpowiedzi używają JSON. Klient korzysta z sesji, a partner z nagłówka `X-API-Key`. Listy mają paginację `page` i `pageSize` (domyślnie 20, maksymalnie 100). Oferta nie ujawnia danych gości.

| Metoda i ścieżka | Działanie | Dostęp |
|---|---|---|
| POST `/auth/register` | Konto klienta | Publiczny |
| POST `/auth/login`, POST `/auth/logout` | Sesja | Publiczny / zalogowany |
| GET/PATCH `/me`, POST `/me/change-password` | Profil i hasło | Zalogowany |
| GET `/rooms`, GET `/packages` | Aktywna oferta | Publiczny lub partner z `catalog:read` |
| GET `/availability?checkIn=...&checkOut=...&guests=...` | Dostępne pokoje i wycena | Publiczny lub partner z `catalog:read` |
| POST `/reservations` | Utworzenie rezerwacji | Klient, personel, partner z `reservations:create` |
| GET `/reservations` | Lista w zakresie uprawnień | Klient, personel, partner z `reservations:read` |
| GET `/reservations/{id}` | Szczegóły | Właściciel/personel lub partner źródłowy |
| PATCH `/reservations/{id}` | Daty i liczba gości | Jak wyżej, partner z `reservations:update` |
| POST `/reservations/{id}/cancel` | Anulowanie | Jak wyżej, partner z `reservations:cancel` |
| POST `/reservations/{id}/check-in` i `/check-out` | Obsługa pobytu | Personel |
| PATCH `/reservations/{id}/items/{itemId}` | Status i notatka realizacji dodatku | Personel |
| GET `/reservations/{id}/plan` | Plan i stan usług | Właściciel/personel lub partner źródłowy z odczytem |
| GET `/reservations/{id}/partner-availability` | Terminy dodatku A/B | Personel |
| POST `/reservations/{id}/partner-bookings` | Rezerwacja wybranego terminu A/B | Personel |
| POST `/reservations/{id}/partner-bookings/{bookingId}/refresh` | Odczyt wyniku/stanu A/B | Personel |
| POST `/reservations/{id}/partner-bookings/{bookingId}/cancel` | Anulowanie w A/B | Personel |
| POST `/reservations/{id}/voucher/check` | Weryfikacja części hotelowej D | Personel |
| POST `/reservations/{id}/voucher/confirm-completion` | Potwierdzenie wykonanego noclegu w D | Personel |
| GET/PATCH `/admin/hotel-settings` | Godziny pobytu | Administrator |
| GET/POST/PATCH `/admin/rooms` i `/admin/packages` z ID dla PATCH | Zarządzanie ofertą | Administrator |
| GET/POST/PATCH `/admin/users` z ID dla PATCH | Konta i role | Administrator |
| POST `/admin/users/{id}/reset-password` | Hasło tymczasowe | Administrator |
| GET/PATCH `/admin/integrations/{partner}` | Konfiguracja A/B/D | Administrator |
| POST `/admin/integrations/{partner}/test` | Test połączenia | Administrator |
| GET/POST `/admin/api-keys`, PATCH `/admin/api-keys/{id}` | Tworzenie i wyłączanie kluczy | Administrator |
| GET `/admin/audit-events` | Historia | Administrator |

Oferta publiczna pokazuje tylko aktywne wpisy. Administrator ma osobne listy obejmujące dezaktywowane wpisy. Nowy klucz otrzymuje tylko wybrane uprawnienia; brak zaznaczenia oznacza brak dostępu.

**Kody odpowiedzi:** 201 utworzenie; 200 odczyt/zmiana; 400 błędne dane; 401 brak/poprawność uwierzytelnienia; 403 brak uprawnienia; 404 brak zasobu lub obcy zasób; 409 konflikt terminu, wersji albo stanu; 429 limit żądań; 503 tymczasowa niedostępność.

Jednolity błąd: `{"error":{"code":"ROOM_UNAVAILABLE","message":"Pokój jest zajęty w wybranym terminie","fields":{}}}`. Nie zwracamy stosu błędu, SQL ani sekretów. Dokumentacja OpenAPI opisuje pola, przykłady, uprawnienia i możliwe błędy każdej operacji.

Tworzenie z D przekazuje referencję vouchera i hotelowej części; C weryfikuje je przez API D. Powiązanie istniejącej rezerwacji z voucherem jest dopuszczone tylko personelowi po weryfikacji, przed zameldowaniem. Klient ani partner nie może dopisać arbitralnie „opłacono voucherem”. Kontrakt transportowy referencji ustalamy z D.


## 10 Przykład sensownej modyfikacji przez API

Partner D może np. zarezerwować pobyt klientowi korzystającemu z jego oferty. Przykładowe identyfikatory poniżej są demonstracyjne.

### Utworzenie

```http
POST /api/v1/reservations
X-API-Key: <klucz przechowywany poza dokumentem>
Idempotency-Key: demo-reservation-001
Content-Type: application/json
```

```json
{
  "roomId": "room-demo-101",
  "checkIn": "2027-06-12",
  "checkOut": "2027-06-15",
  "guests": 2,
  "expectedTotalMinor": 90000,
  "packageId": "package-demo-relax",
  "guest": {"name": "Jan Testowy", "email": "jan@example.com", "phone": "+48123456789"},
  "guestNames": ["Jan Testowy", "Anna Testowa"]
}
```

Przy stawce 250 zł/noc i pakiecie 150 zł odpowiedź zawiera m.in.:

```json
{
  "id": "reservation-demo-001",
  "number": "C-DEMO-001",
  "status": "confirmed",
  "checkIn": "2027-06-12",
  "checkOut": "2027-06-15",
  "guests": 2,
  "pricing": {"nights": 3, "nightlyRateMinor": 25000, "packagePriceMinor": 15000, "totalMinor": 90000, "currency": "PLN"},
  "version": 1
}
```

### Zmiana terminu

```http
PATCH /api/v1/reservations/reservation-demo-001
X-API-Key: <klucz partnera źródłowego>
Content-Type: application/json
```

```json
{"checkIn":"2027-06-13","checkOut":"2027-06-16","guests":2,"expectedVersion":1,"expectedTotalMinor":90000}
```

Partner najpierw pobiera dostępność i aktualną wycenę. Serwer ponownie ją sprawdza. Jeśli kwota jest inna niż zaakceptowana `expectedTotalMinor`, zwraca 409 z nową wyceną i nie zmienia rezerwacji. Nieaktualna `expectedVersion` również daje 409; partner odczytuje najnowszy stan. Cena nie jest narzucana przez partnera.

`Idempotency-Key` jest wymagany przy tworzeniu rezerwacji przez partnera i stosowany także przez formularz klienta. Ten sam klucz i ta sama treść zwracają wcześniejszy wynik; zmieniona treść pod tym samym kluczem daje 409. Rekordy przechowujemy przez cały cykl demonstracyjny projektu. Ponowne anulowanie już anulowanej rezerwacji zwraca jej aktualny stan, bez kolejnego skutku biznesowego.

API nie pozwala dowolnie zmieniać ceny, partnera źródłowego, właściciela, pokoju, pakietu lub statusu przez `PATCH`. Nieobsługiwane pola są odrzucane. Dla tworzenia również wymagamy zaakceptowanej kwoty `expectedTotalMinor`. Przykład pokazuje bezpośrednią rezerwację partnera; wariant wspólnego scenariusza z D dodatkowo przekazuje `voucherReference` i `hotelComponentId`, których istnienie oraz zakres C sprawdza w D. Cena referencyjna nie oznacza obciążenia klienta drugi raz.


## 11 Konfigurowalne integracje A B D

Rozdzielamy dostęp partnerów do C oraz połączenia C z API partnerów. Administrator konfiguruje stałe adaptery, nie buduje dowolnych mapowań JSON.

| Ustawienie | Zasada |
|---|---|
| Adres bazowy | Osobny dla A/B/D; HTTP/HTTPS; zmiana bez przebudowy |
| Tryb | Demo/real, jawnie widoczny; symulator realizuje uzgodniony kontrakt |
| Włącznik | Blokuje nowe wywołania; wcześniej zapisane referencje pozostają |
| Uwierzytelnienie wychodzące | Klucz/token zgodnie z kontraktem; zapis szyfrowany, prezentacja maskowana |
| Timeout | Domyślnie 5 s, konfigurowalny 1–30 s |
| Test połączenia | Operacja odczytowa, czas i wynik ostatniego testu |
| Klucze wejściowe C | Osobne per partner, hash w bazie, wartość widoczna raz, wyłączanie i wymiana |
| Uprawnienia wejściowe | catalog:read, reservations:create/read/update/cancel; wyłącznie własne pobyty |

Adresy sprawdza backend względem listy dopuszczonych hostów środowiska. Blokuje niejawne adresy lokalne/metadane chmurowe i przekierowania omijające ograniczenie. Hosty lokalnych symulatorów mogą być jawnie dopuszczone.

### Minimalne możliwości API partnerów wymagane przez C

| Partner | Wywołania C | Dane odpowiedzi |
|---|---|---|
| A | Katalog, terminy, utworzenie rezerwacji, odczyt jej stanu, anulowanie, odczyt wyniku operacji | Oferta, początek/koniec, ID i status rezerwacji, wynik dla ID operacji |
| B | Katalog treningów, wolne terminy, zapis, odczyt stanu, anulowanie, odczyt wyniku operacji | Oferta, termin, dostępna liczba miejsc, ID i status zapisu |
| D | Weryfikacja części hotelowej, potwierdzenie wykonania, odczyt wyniku operacji | Ważność, zakres, dostawca, ID części, wykorzystanie i referencja rezerwacji |

Dokładne ścieżki i nazwy pól partnerów nie są narzucone slajdami. Wiktoria i Adrian uzgadniają je z zespołami, a adapter tłumaczy je na model C. Wszystkie zmieniające dane wywołania mają unikalne ID operacji i uzgodnioną idempotencję. A/B blokują swoje zasoby u siebie; C nie gwarantuje dostępności na podstawie starej listy terminów.

Terminy wybiera recepcja po rozmowie z klientem. Nie budujemy automatycznego algorytmu układania całego planu ani jednej transakcji obejmującej wszystkie systemy. Potwierdzenie jednej usługi i odmowa drugiej tworzą jawnie częściowo potwierdzony plan.


## 12 Stany integracji i obsługa niepowodzeń

**Pozycja planu A/B:** `unplanned → pending → confirmed → completed`, z alternatywami `rejected`, `unknown`, `cancel_pending`, `cancelled`. Zakończenie usługi pochodzi z A/B przy odświeżeniu, nie z dowolnej edycji statusu w C.

**Potwierdzenie hotelowej części vouchera:** `not_reported → pending → reported`, alternatywnie `rejected` lub `unknown`. Status pobytu hotelowego oraz stan przekazania wyniku do D są osobne. Wymeldowanie może być zapisane mimo awarii D; interfejs wtedy jasno pokazuje „pobyt zakończony, potwierdzenie D oczekuje”.

1. Przed zewnętrzną zmianą C zapisuje operację z unikalnym ID.
2. Jednoznaczny sukces zapisuje wynik i zewnętrzny identyfikator.
3. Jednoznaczna odmowa nie tworzy potwierdzonej pozycji.
4. Timeout po wysłaniu zachowuje `unknown`; nie wysyłamy nowego żądania z nowym ID.
5. Personel sprawdza wynik u partnera lub ponawia bezpiecznie tę samą operację zgodnie z kontraktem.
6. Po restarcie C zapisane operacje pozostają widoczne do wyjaśnienia. Dla tej samej pozycji blokujemy kolejną operację tworzenia, dopóki wynik jest niepewny.

| Sytuacja | Reakcja |
|---|---|
| Masaż potwierdzony, trening odrzucony | Zachować masaż i hotel; plan częściowy; recepcja wybiera inny trening lub świadomie anuluje masaż |
| Usługa wysłana, timeout odpowiedzi | Zachować ID operacji; ustalić wynik przed kolejnym zapisem |
| Anulowanie masażu udało się, treningu nie | Hotel pozostaje aktywny; częściowe anulowania widoczne; ponowić/wyjaśnić trening |
| D nie działa przy zameldowaniu voucherowym | Brak automatycznej akceptacji; personel wyjaśnia; zwykłe pobyty bez D działają |
| D nie działa po wymeldowaniu | Zakończony pobyt i osobny status wysłania; bez podwójnego wykorzystania |
| Partner zwraca błędne dane | Odrzucić odpowiedź, pokazać błąd, zapisać bezpieczną diagnostykę |
| Voucher nie obejmuje noclegu C / część zużyta | Odrzucić operację voucherową; nie zużywać A/B |

Odczyty ponawiamy maksymalnie raz przy przejściowym błędzie. Nie dodajemy kolejki ani automatycznego systemu napraw: wystarczą trwały rejestr operacji i przyciski odświeżenia/ponowienia w recepcji.

**Do uzgodnienia z partnerami przed realną integracją:** uwierzytelnienie, ID operacji i odczyt wyniku, format dat ze strefą, statusy, identyfikatory ofert/rezerwacji/części vouchera, zakres danych gościa, błędy i anulowanie. Jeśli partner nie zapewnia potrzebnej możliwości, wpisujemy blokującą zależność i uzgadniamy rozwiązanie; symulator nie zamyka końcowego odbioru tej integracji.


## 13 Podział pracy i odpowiedzialność

Osoba odpowiedzialna za dany obszar sprawdza kod, testy i dokumentację oraz przedstawia wynik na odbiorze.

| Osoba | Rola wiodąca | Konkretny rezultat | Proponowany backup |
|---|---|---|---|
| Adrian Zieliński | Koordynator | Tablica zadań, etapy, rejestr ryzyk, uzgodnienia A/B/D, instrukcja uruchomienia i wspólna demonstracja | Wiktoria |
| Piotr Świadek | Przedstawiciel klienta | Priorytety F-01–F-13, akceptacja scenariuszy S-01–S-06 i raport odbioru | Adrian |
| Wiktoria Sobejko | Analityk i projektant | Model danych, reguły, makiety, OpenAPI i protokół ustaleń partnerów | Andrii |
| Andrii Yaremko | Backend | Baza/migracje, konta, rezerwacje, role, API, adaptery i testy logiki | Filip |
| Rafał Świętkowski | Interfejs | Ekrany, formularze, nawigacja, połączenie z API i komunikaty błędów | Wiktoria |
| Filip Włodarczyk | Testy automatyczne | Testy API, kontraktów, współbieżności i E2E; uruchamianie kontroli w CI | Andrii |
| Jesika Proczek | Testy manualne | Scenariusze, raport testów, zgłoszenia z krokami i potwierdzenia poprawek | Piotr |

Adrian przejmuje dodatkową rolę DevOps/integratora, ze wsparciem Filipa w automatyzacji testów. Backupy są propozycją do osobistego potwierdzenia w rewizji; nie oznaczają równoległego tworzenia całego modułu.

Każde zadanie ma właściciela, opis wyniku, kryterium odbioru i status: `do zrobienia → w toku → do weryfikacji → gotowe`. Zadanie zablokowane opisuje przeszkodę i potrzebną decyzję. Przy poprawkach używamy małych gałęzi i przeglądu zmian.


**Wymagania organizacyjne z prezentacji 1, slajdy 16–18:** własne repo GitHub zespołu C, dostęp dla członków i prowadzącego, zmiany każdego studenta na jego własnym koncie, osobne gałęzie i pull requesty. Teams służy do rozmów, GitHub Issues/Projects do zadań i błędów. Zmianę kontraktu API uzgadniamy z konsumentami przed jej wdrożeniem.

Każda osoba prowadzi krótki zapis wkładu: zadanie, rezultat lub link do PR albo dokumentu, sposób weryfikacji i test. Na obronie wyjaśnia własne rozwiązanie.


## 14 Iteracje i harmonogram względny

Stosujemy krótkie przyrosty: działający fragment, test, pokaz Piotrowi i korekta backlogu. To praktyczne zastosowanie podejścia iteracyjnego z prezentacji 2; nie odkładamy integracji i testów na koniec. Nie deklarujemy formalnego Scrum.

**Plan bazowy do akceptacji:** sześć iteracji po jednym tygodniu od wspólnie ustalonego startu T0. Jest to założenie planistyczne, nie termin narzucony przez prowadzącego. Adrian wpisuje konkretne daty po poznaniu terminu zaliczenia. Każde issue otrzymuje termin najpóźniej końca przypisanej iteracji.

| Iteracja | Prace i właściciele | Kamień milowy |
|---|---|---|
| I1 | Piotr/Wiktoria: zakres, model, makiety i kontrakty; Adrian: repo, plan integracji; Andrii/Rafał: szkielet | M1: zatwierdzony rdzeń C, szkic OpenAPI przekazany D, symulatory kontraktów |
| I2 | Andrii/Rafał: konta, pokoje, dostępność, rezerwacja; Filip/Jesika: walidacja i konflikt | M2: prototyp z własną bazą i API; D może wykonać próbne wywołanie |
| I3 | Zmiana/anulowanie, pakiety, goście, recepcja, klucze; adapter A; weryfikacja zespołu | M3: pierwsze rzeczywiste wywołanie API partnera; sześć użytecznych operacji C |
| I4 | Andrii/Rafał: plan pobytu, A/B rezerwacje i anulowania; Filip: częściowe błędy | M4: D tworzy pobyt, C potwierdza usługi A/B |
| I5 | Voucher częściowy D, realizacja noclegu, idempotencja, awarie i wspólna próba | M5: wersja beta całego scenariusza A/B/C/D |
| I6 | Regresja, poprawki, dokumentacja, logi scenariusza, zapis wkładu, próbna obrona | M6: wydanie i raport odbioru |

**Zależności:** API C dla D do M2; kontrakty A/B/D do M1; rzeczywiste API A dla pierwszej integracji do M3; A/B dla planu do M4; D dla weryfikacji i wykonania części do M5. Terminy partnerów wymagają ich potwierdzenia w Teams i rejestrze ustaleń.

| Praca / tydzień względny | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| Ustalenia i kontrakty | ● | ● | aktualizacja | aktualizacja | aktualizacja | odbiór |
| Hotel i własne API | szkielet | ● | ● | poprawki | poprawki | odbiór |
| Integracje | kontrakty | próby | A | A/B | D + całość | odbiór |
| Testy i dokumentacja | ● | ● | ● | ● | ● | ● |

Raz w tygodniu: krótki przegląd postępu i ryzyk, pokaz przyrostu, aktualizacja zadań/terminów. Opóźnienie partnera zgłaszamy od razu; lokalnie kontynuujemy na symulatorze i umawiamy termin ponownej integracji.

Definicja gotowego zadania: wynik spełnia kryteria, backend waliduje dane i uprawnienia, błędy są obsłużone, testy przechodzą, dokumentacja jest aktualna, PR został sprawdzony i rezultat odebrany. Nową funkcję spoza zakresu wpisujemy do rejestru zmiany wraz z wpływem na termin i partnerów.


## 15 Testy i scenariusz prezentacji

| ID | Test | Oczekiwany wynik | Odpowiedzialność |
|---|---|---|---|
| T-01 | Rejestracja, duplikat email, logowanie, blokada | Poprawne konto; duplikat odrzucony; blokada kończy sesję | Filip / Jesika |
| T-02 | Klient próbuje nadać sobie administratora | Operacja odrzucona | Filip |
| T-03 | Daty, pojemność, brak dostępności | Błędne dane odrzucone; pusty wynik poprawnie pokazany | Filip / Jesika |
| T-04 | Dwie jednoczesne rezerwacje tego samego pokoju | Dokładnie jedna rezerwacja utworzona; druga otrzymuje konflikt | Filip |
| T-05 | Wyjazd jednego i przyjazd drugiego tego samego dnia | Obie rezerwacje dozwolone | Filip |
| T-06 | Cena 3 nocy i pakietu; późniejsza zmiana cennika | Poprawna suma; poprzedni snapshot ceny zachowany | Filip |
| T-07 | Zmiana na zajęty termin i nieaktualna wersja/cena | Brak częściowej zmiany; odpowiedź 409 | Filip |
| T-08 | Odczyt/anulowanie obcego pobytu przez klienta lub partnera | Brak danych i zmiany | Filip |
| T-09 | Ponowione POST z tym samym kluczem idempotencji | Jedna rezerwacja; inna treść pod tym kluczem odrzucona | Filip |
| T-10 | Odebranie zakresu i wyłączenie klucza API | Kolejne żądanie zablokowane bez restartu | Filip / Jesika |
| T-11 | Zameldowanie/wymeldowanie w niewłaściwym stanie | Odrzucone; poprawne przejścia działają | Filip / Jesika |
| T-12 | Pakiet dezaktywowany lub zmieniony | Nowy zakup zablokowany; stary pobyt zachowuje dodatki | Filip |
| T-13 | A/B/D timeout, błędny JSON, niewłaściwy klucz | Czytelny komunikat, brak sekretów, zwykły pobyt działa | Filip |
| T-14 | Część hotelowa D: sukces, odmowa, timeout po sukcesie, restart C | Nocleg potwierdzony raz; masaż/trening nie zużyte przez C | Filip |
| T-15 | Zmiana adresu/trybu integracji | Działa bez przebudowy; tryb demo widoczny | Jesika |
| T-16 | Ostatni administrator, reset hasła i CSRF | Nie ma utraty administracji ani nieuprawnionej zmiany | Filip |
| T-17 | Formularze, klawiatura, telefon, puste listy | Czytelna i używalna obsługa | Jesika |
| T-18 | Uruchomienie według README i ponowienie seed | Aplikacja startuje; dane istniejące nie giną | Adrian / Filip |

Automatyzujemy testy reguł i API, a E2E ograniczamy do rezerwacji klienta oraz obsługi recepcji. Nie tworzymy testów jedynie powtarzających wygląd każdego komponentu. CI sprawdza typy, testy i budowanie aplikacji.


### Dodatkowe testy integracyjne wymagane wspólnym scenariuszem

| ID | Scenariusz | Wynik |
|---|---|---|
| T-19 | D tworzy pobyt z referencją vouchera i powtarza żądanie | Jedna rezerwacja, poprawne źródło i powiązanie |
| T-20 | C wybiera A/B poza pobytem lub nakładające się usługi | Odrzucone przed wysłaniem do partnera |
| T-21 | Masaż sukces, trening odmowa/brak miejsca | Częściowy plan; hotel i masaż nie giną |
| T-22 | Timeout A/B po utworzeniu i restart C | Odczyt wyniku po ID; brak kolejnej rezerwacji |
| T-23 | Zmiana/anulowanie pobytu z aktywnymi A/B | Brak osieroconych rezerwacji; częściowe anulowania widoczne |
| T-24 | Voucher nieważny, obcy dostawca, niewłaściwa część | Brak zameldowania voucherowego i brak wykorzystania |
| T-25 | Pobyt zakończony, D chwilowo niedostępne | Zakończenie lokalne zachowane; jedno potwierdzenie po odzyskaniu połączenia |
| T-26 | Stan usługi zmienił się w A/B | Odświeżenie aktualizuje plan C; nie konsumuje jej części w D |

### Demonstracja wspólna około 10–15 minut

1. D sprzedaje demonstracyjny voucher: nocleg, masaż, trening i przez API sprawdza ofertę/dostępność C.
2. D tworzy pobyt w C; C pokazuje go w recepcji z gościem, pakietem i referencją vouchera.
3. C uzgadnia terminy, wywołuje A/B i prezentuje zewnętrzne potwierdzenia na planie.
4. Pokazujemy konflikt terminu lub odmowę treningu i wybór alternatywy.
5. C sprawdza część noclegową w D, zameldowuje, a po pobycie wymeldowuje i potwierdza wykonanie tej części.
6. A/B we własnych aplikacjach sprawdzają voucher, kończą usługi i potwierdzają swoje części. D pokazuje pozostałe usługi i zestawienie.
7. C pokazuje rzeczywistą zmianę rezerwacji przez API, wyłączenie uprawnienia oraz historię; osobno krótki własny pobyt bez vouchera.

Przebieg rejestrujemy bez sekretów: ID pobytu, ID usług A/B, referencja części D, wyniki wywołań i raport testów. Tryb demo jest jawny; finalny wspólny scenariusz musi potwierdzić współpracę rzeczywistych aplikacji.

**Odbiór końcowy według prezentacji 1, slajdy 18–19:** uruchamialna aplikacja C z własną bazą; co najmniej pięć użytecznych operacji API (oferta, dostępność, utworzenie, odczyt, zmiana, anulowanie); działające wywołania A/B/D dla scenariusza; testy poprawnej wymiany, błędnych danych, konfliktów i niedostępności; dokumentacja API/uruchomienia; wspólna demonstracja; zapis wkładu i weryfikacji pracy każdej osoby. Nie ma błędów pozwalających przejąć rolę, odczytać cudze dane lub podwójnie zarezerwować pokój/usługę.


## 16 Rejestr uwag i ryzyk

Uwagi mają wagę: **blokująca** (uniemożliwia akceptację), **istotna** (wymaga decyzji), **drobna** (redakcja lub ulepszenie). Statusy: nowe, analizowane, przyjęte, odrzucone z uzasadnieniem, wdrożone, zweryfikowane.

| ID uwagi | Autor | Sekcja / wymaganie | Waga | Problem i proponowana zmiana | Odpowiedzialny | Status / decyzja |
|---|---|---|---|---|---|---|
| U-001 | — | — | — | Miejsce na pierwszą uwagę | — | — |
| U-002 | — | — | — | Miejsce na kolejną uwagę | — | — |

### Ryzyka początkowe

| ID | Ryzyko | Działanie i właściciel |
|---|---|---|
| R-01 | Partnerzy mają inne kontrakty lub późno udostępniają API | Wczesne uzgodnienie; adaptery i symulatory; Adrian/Wiktoria |
| R-02 | A/B/D nie zapewnia idempotencji lub odczytu wyniku | Zachować stan niepewny; nie duplikować żądania; uzgodnić kontrakt; Adrian/Andrii |
| R-03 | Niespójności lub błędy bezpieczeństwa w kodzie | Małe zadania, przegląd, testy uprawnień i współbieżności; Andrii/Filip |
| R-04 | Rozrost zakresu | Nowe funkcje dopiero po rdzeniu i decyzji Piotra; Adrian |
| R-05 | SQLite blokuje się przy zapisach | Krótkie transakcje, kontrolowane ponowienia i test równoległy; Andrii |
| R-06 | Brak udokumentowanego wkładu i weryfikacji pracy | Każda rola odbiera swój rezultat i potrafi go zademonstrować; cały zespół |

Jeśli ograniczenia SQLite uniemożliwią spełnienie testów, migracja do PostgreSQL wymaga osobnej uzgodnionej rewizji; nie jest automatycznym rozszerzeniem zakresu.


## 17 Rejestr decyzji i wersji

| ID | Temat | Proponowane ustalenie | Status / zatwierdzający |
|---|---|---|---|
| D-01 | Architektura | React/Express/TypeScript/Prisma/SQLite, jedna instancja | Do akceptacji: Wiktoria, Andrii, Rafał, Filip |
| D-02 | Zakres | Jeden hotel; własni klienci i pełny scenariusz A/B/C/D | Do akceptacji: Piotr i cały zespół |
| D-03 | Konfiguracja | Panel i stałe adaptery A/B/D | Do akceptacji: Wiktoria, Andrii, Adrian |
| D-04 | Partnerzy | D rezerwuje C; C rezerwuje A/B; każdy realizuje swoją część | Wymóg ze slajdów; kontrakty do uzgodnienia |
| D-05 | Voucher | Voucher usługowy/częściowy; C potwierdza tylko nocleg po wykonaniu | Wymóg scenariusza; szczegóły do uzgodnienia z D |
| D-06 | Role | Adrian integrator, Filip wspiera CI, backupy według sekcji 13 | Do akceptacji całego zespołu |
| D-07 | Plan | 6 iteracji tygodniowych; konkretne daty po ustaleniu T0 i zaliczenia | Założenie do akceptacji: Adrian/zespół |
| D-08 | Częściowe błędy | Jawne stany i ręczne wyjaśnienie, brak transakcji globalnej | Do akceptacji: Wiktoria, Andrii, Filip |

| Wersja | Data | Zakres zmian | Podstawa |
|---|---|---|---|
| 0.1 | 11.10.2026 | Wstępny plan rozmowy; bez akceptacji zespołu | Pierwsze ustalenia użytkownika |
| 0.2 | 11.10.2026 | Rezerwacje A/B, plan pobytu, voucher częściowy, wymogi zaliczenia, iteracje i dokumentowanie wkładu | Obie prezentacje i wspólny scenariusz |
| 0.3 | — | Miejsce na rewizję zespołu | Uwagi U-… |

Zmianę odrzuconą uzasadniamy. Zamknięcie uwagi nie zatwierdza całego dokumentu. Decyzje partnerów zapisujemy z datą, osobą potwierdzającą i odnośnikiem do kontraktu/issue. Zaakceptowany kontrakt i finalny plan trafiają również do repozytorium C.


## 17a Powiązanie planu z materiałami prowadzącego

| Materiał | Wymaganie lub wskazówka | Realizacja w planie |
|---|---|---|
| Prezentacja 1, slajdy 4 i 19 | Niezależna baza, własne API i klient API partnera | Sekcje 7–12; brak bezpośredniego dostępu do baz A/B/D |
| Prezentacja 1, slajd 11 | Pokoje/wyposażenie, pobyty, pakiety, liczby usług i koordynacja | F-02–F-07, F-11, plan pobytu |
| Prezentacja 1, slajd 13 i dostarczone zdjęcie | D → C → A/B → realizacja części w D | S-03, sekcje 11–12, demonstracja |
| Prezentacja 1, slajdy 6–7 | Prototyp, pierwsza integracja, beta, wspólny harmonogram | M1–M6 i terminy zależności partnerów |
| Prezentacja 1, slajdy 14–17 | Role, repo, zadania, PR, Teams, kontrakty | Sekcje 13–14 i rejestry uwag/decyzji |
| Prezentacja 1, slajdy 18–19 | ≥5 operacji, testy wymiany i awarii, zapis wkładu | Sekcja 15 i indywidualny zapis wkładu |
| Prezentacja 2, slajd 4 | Widoczny harmonogram i zależności | Tabela tygodni względnych, kamienie milowe |
| Prezentacja 2, slajdy 8–13 | Korekta planu, iteracje, udział klienta i przyrosty | Cotygodniowy pokaz Piotrowi, backlog i rewizje |

### Materiały źródłowe

- `Projekt zespołowy_prezentacja_1.pptx` — 22 slajdy; zakres projektów, integracja, organizacja i kryteria zaliczenia.
- `Projekt zespołowy_prezentacja_2.pptx` — 14 slajdów; planowanie, zależności i podejście iteracyjne.
- Zdjęcia przekazane w rozmowie — skład projektu C, role i wspólny scenariusz.

Prezentacje należy przechowywać w uzgodnionym folderze zespołu, a w finalnej rewizji dopisać jego rzeczywisty link. Ten notebook nie zawiera osadzonych kopii PPTX.


## 18 Akceptacja zespołu

Dozwolone wpisy: **oczekuje**, **akceptuję**, **akceptuję warunkowo**, **wymaga zmian**. Akceptacja warunkowa zawiera identyfikatory uwag i nie zamyka wersji, dopóki warunki nie są spełnione.

| Osoba | Wersja przeczytana | Decyzja | Data | Warunki / uwagi / potwierdzenie backupu |
|---|---|---|---|---|
| Adrian Zieliński | — | oczekuje | — | — |
| Piotr Świadek | — | oczekuje | — | — |
| Rafał Świętkowski | — | oczekuje | — | — |
| Jesika Proczek | — | oczekuje | — | — |
| Wiktoria Sobejko | — | oczekuje | — | — |
| Andrii Yaremko | — | oczekuje | — | — |
| Filip Włodarczyk | — | oczekuje | — | — |

**Protokół zamknięcia rewizji — uzupełnia Adrian:**

- Zatwierdzona wersja: —
- Data: —
- Blokujące uwagi rozwiązane: —
- Zakres potwierdzony przez Piotra: —
- Kontrakty rzeczywiste A/B/D i nadal otwarte zależności: —
- Termin zaliczenia i terminy etapów: —
- Gotowość do rozpoczęcia kolejnego etapu: —
